# Generating Psalm- and Proverb-Style Text in Spanish with a Character-Level GPT Trained from Scratch

**Name:** Arturo Ramos
**Dataset:** *Santa Biblia — Reina Valera 1909* (public domain), plain-text verse-per-line edition from eBible.org — https://ebible.org/find/details.php?id=spaRV1909 — file `data/spaRV1909_vpl.txt`

**Generative task.** This is **Transformer-based text generation**. A small decoder-only Transformer (a GPT-style language model) is trained from scratch, character by character, to generate new Spanish text in the style of the poetic and wisdom books of the 1909 Reina-Valera Bible, the Psalms (*Salmos*) and Proverbs (*Proverbios*). Training has two phases: pre-training on the whole Bible teaches the model 1909 Spanish, and fine-tuning on Psalms and Proverbs teaches it their style. Each verse is prefixed with its book code (`PSA:` or `PRO:`), so the prefix works as a prompt that asks for one style or the other. The system produces short verse-like lines; it is a study of how a generative model imitates a sacred text, not a tool for producing scripture.

## 1. Setup

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # required for deterministic cuBLAS kernels

import math
import random
import re
import time
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 140)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_FILE = Path("data") / "spaRV1909_vpl.txt"
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
print("torch", torch.__version__, "| device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")

## 2. Load and Inspect the Data

The eBible.org "VPL" edition has one verse per line in the form `BOOK chapter:verse text`, with all headings, notes and paragraph marks removed.

In [ ]:
lines = DATA_FILE.read_text(encoding="utf-8-sig").splitlines()
pattern = re.compile(r"^(\w{3}) (\d+):(\d+) (.*)$")
parsed = [pattern.match(l) for l in lines]
print("lines:", len(lines), "| lines that do not match 'BOOK c:v text':", sum(p is None for p in parsed))

verses = pd.DataFrame([p.groups() for p in parsed], columns=["book", "chapter", "verse", "text"])
verses["chapter"] = verses["chapter"].astype(int)
verses["verse"] = verses["verse"].astype(int)
verses["chars"] = verses["text"].str.len()
print("books:", verses["book"].nunique(), "| total characters:", f"{verses['chars'].sum():,}")
verses.head()

In [ ]:
# Representative samples: the two target books and a narrative book for contrast
pd.concat([verses[(verses.book == "PSA") & (verses.chapter == 23)].head(3),
           verses[(verses.book == "PRO") & (verses.chapter == 3)].iloc[4:7],
           verses[(verses.book == "GEN") & (verses.chapter == 1)].head(2)])[["book", "chapter", "verse", "text"]]

In [ ]:
target = verses[verses["book"].isin(["PSA", "PRO"])]
summary = pd.DataFrame({
    "verses": [len(verses), len(target), (verses.book == "PSA").sum(), (verses.book == "PRO").sum()],
    "characters": [verses.chars.sum(), target.chars.sum(), verses.loc[verses.book == "PSA", "chars"].sum(), verses.loc[verses.book == "PRO", "chars"].sum()],
    "median characters per verse": [verses.chars.median(), target.chars.median(), verses.loc[verses.book == "PSA", "chars"].median(), verses.loc[verses.book == "PRO", "chars"].median()],
}, index=["whole Bible", "Psalms + Proverbs", "Psalms", "Proverbs"])
summary

In [ ]:
# Character inventory and orthographic features of the 1909 text
all_text = "".join(verses["text"])
char_counts = Counter(all_text)
print("distinct characters:", len(char_counts))
print("".join(sorted(char_counts)))
features = {
    "verses with square brackets [ ]": int(verses["text"].str.contains(r"\[").sum()),
    "occurrences of the preposition 'á'": len(re.findall(r"\bá\b", all_text)),
    "occurrences of 'fué'": len(re.findall(r"\bfué\b", all_text)),
    "Psalm verses that start with a heading ('Salmo de ...')": int(verses[(verses.book == "PSA")]["text"].str.match(r"^(Salmo|Al Músico|Masquil|Mictam|Oración)").sum()),
    "verses whose first word is fully upper-case": int(verses["text"].str.match(r"^[A-ZÁÉÍÓÚÑ]{2,}\b").sum()),
}
pd.Series(features, name="count")

**Structure, formats and preprocessing considerations**

- The file has 31,102 verse lines from the 66 books, about 3.8 million characters in total and only 83 distinct characters, which makes a character-level model small and practical: the vocabulary fits in less than a hundred symbols and no word can be out of vocabulary.
- The two target books are a small part of the corpus: Psalms and Proverbs together have 3,376 verses (about 7 % of the characters). A model trained only on them would see too little Spanish to learn spelling and grammar, which motivates pre-training on the whole Bible first.
- The text keeps the 1909 orthography (for example the preposition *á* written with an accent and *fué* with an accent), which is part of the style to be imitated, so it is **not** modernized.
- Square brackets mark words added by the translators for clarity. They are editorial marks rather than language, so the brackets are removed and the words kept.
- Some verses start with a fully upper-case word (the first word of a chapter, as in *EN el principio*), and the first verse of many Psalms contains its heading (*Salmo de David.*). Both are kept: they are part of how the text looks, and the model will reproduce them.

## 3. Preprocessing and Data Splits

In [ ]:
def clean_verse(text: str) -> str:
    """Normalize one verse: Unicode NFC, remove the translators' square brackets, collapse whitespace."""
    text = unicodedata.normalize("NFC", text).replace("[", "").replace("]", "")
    return re.sub(r"\s+", " ", text).strip()


def to_training_line(book: str, text: str) -> str:
    """Format a verse as one training line: the book code works as a style prompt, the newline ends the verse."""
    return f"{book}: {text}\n"


verses["clean"] = verses["text"].map(clean_verse)

# Split by CHAPTER so that validation verses never share a chapter with training verses
chapters = sorted(set(zip(verses.book, verses.chapter)))
random.Random(SEED).shuffle(chapters)
val_chapters = set(chapters[: int(0.10 * len(chapters))])
verses["split"] = ["val" if (b, c) in val_chapters else "train" for b, c in zip(verses.book, verses.chapter)]

def corpus(frame: pd.DataFrame) -> str:
    return "".join(to_training_line(b, t) for b, t in zip(frame["book"], frame["clean"]))

is_target = verses["book"].isin(["PSA", "PRO"])
texts = {
    "pretrain_train": corpus(verses[verses.split == "train"]),
    "pretrain_val": corpus(verses[verses.split == "val"]),
    "finetune_train": corpus(verses[(verses.split == "train") & is_target]),
    "finetune_val": corpus(verses[(verses.split == "val") & is_target]),
}
pd.DataFrame({"characters": {k: len(v) for k, v in texts.items()},
              "lines (verses)": {k: v.count("\n") for k, v in texts.items()}})

In [ ]:
# Character vocabulary (from the training text) and encoding
VOCAB = sorted(set(texts["pretrain_train"]))
STOI = {c: i for i, c in enumerate(VOCAB)}
ITOS = {i: c for c, i in STOI.items()}
missing = set(texts["pretrain_val"]) - set(VOCAB)
print("vocabulary size:", len(VOCAB), "| characters in validation but not in training:", missing or "none")

def encode(s: str) -> torch.Tensor:
    return torch.tensor([STOI[c] for c in s], dtype=torch.long)

def decode(ids) -> str:
    return "".join(ITOS[int(i)] for i in ids)

data = {k: encode(v) for k, v in texts.items()}
print({k: tuple(v.shape) for k, v in data.items()})
print("example line:", repr(texts["finetune_train"][:120]))

**What was done and why**

- **Cleaning** is deliberately minimal: Unicode normalization, removal of the translators' brackets and whitespace collapsing. The archaic spelling is the target style and is left untouched.
- **Training format.** Each verse becomes one line, `PSA: …` or `PRO: …` (or the code of any other book during pre-training). The book code is a lightweight *conditioning prompt*: at generation time, starting from `PRO: ` asks the model for a proverb-style line. The newline marks the end of a generated verse.
- **Chapter-level split (90 / 10 %).** Verses of the same chapter share wording and repetitions (the refrain of Psalm 136 appears in every verse), so validation chapters are held out entirely. The fine-tuning splits are the Psalms and Proverbs chapters of the same two splits, so no validation chapter is ever used for training in either phase.
- **Character vocabulary** of the training text; every validation character is covered.

## 4. Model: a Character-Level GPT

The model is a decoder-only Transformer, the architecture of the GPT family, written from scratch:

- **Token and position embeddings** (dimension 384, context of 192 characters).
- **6 Transformer blocks**, each with pre-layer-normalization, **causal multi-head self-attention** (6 heads) and a feed-forward network (4 × 384 with GELU), both with residual connections and dropout 0.1. The causal mask lets each position attend only to earlier characters, which is what makes the model generative: it learns the probability of the next character given all previous ones.
- **Output head**: a final layer norm and a linear layer to the character vocabulary; the input embedding and output weights are tied.
- **Loss**: cross-entropy of the next character (equivalently, maximum likelihood of the text).

In [ ]:
class CausalSelfAttention(nn.Module):
    """Multi-head self-attention in which each position only sees itself and earlier positions."""

    def __init__(self, d_model: int, n_heads: int, block_size: int, dropout: float):
        super().__init__()
        self.n_heads = n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size, dtype=torch.bool)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))
        att = att.masked_fill(~self.mask[:T, :T], float("-inf"))
        att = self.attn_drop(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y))


class Block(nn.Module):
    """Pre-norm Transformer block: causal self-attention and a feed-forward network, each with a residual connection."""

    def __init__(self, d_model: int, n_heads: int, block_size: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model), nn.Dropout(dropout))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))


class CharGPT(nn.Module):
    """Decoder-only Transformer language model over characters."""

    def __init__(self, vocab_size: int, block_size: int = 192, d_model: int = 384, n_layers: int = 6, n_heads: int = 6, dropout: float = 0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, block_size, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight          # weight tying
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        T = idx.size(1)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device)))
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))


BLOCK_SIZE = 192
model = CharGPT(len(VOCAB), block_size=BLOCK_SIZE).to(DEVICE)
print(f"trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

### Training setup

- **Objective:** next-character cross-entropy on random windows of 192 characters (batch of 64 windows).
- **Optimizer:** AdamW (weight decay 0.1), gradient clipping at 1.0, learning rate with a linear warm-up of 200 steps followed by cosine decay.
- **Phase 1 — pre-training:** 4,000 steps on the whole Bible, peak learning rate 1e-3.
- **Phase 2 — fine-tuning:** up to 600 more steps on Psalms and Proverbs only, with a lower peak learning rate of 1e-4 so that the style is adapted without erasing what was learned. The fine-tuning text is small (269,000 characters, so 600 steps show it to the model about 27 times), which makes overfitting likely; the loss is therefore checked every 25 steps and **early stopping** keeps the weights with the lowest validation loss. The weights at the end of the run are also kept, to study what overfitting does to the generated text.
- The loss is estimated on fixed batches of the training and validation text of the current phase (every 250 steps in pre-training, every 25 in fine-tuning).

In [ ]:
BATCH_SIZE = 64
EVAL_BATCHES = 20


def get_batch(split_data: torch.Tensor, generator: torch.Generator) -> tuple[torch.Tensor, torch.Tensor]:
    """Sample BATCH_SIZE random windows of BLOCK_SIZE characters and their next-character targets."""
    ix = torch.randint(len(split_data) - BLOCK_SIZE - 1, (BATCH_SIZE,), generator=generator)
    x = torch.stack([split_data[i:i + BLOCK_SIZE] for i in ix])
    y = torch.stack([split_data[i + 1:i + BLOCK_SIZE + 1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)


@torch.no_grad()
def estimate_loss(model: nn.Module, train_data: torch.Tensor, val_data: torch.Tensor) -> dict:
    """Mean cross-entropy on the same fixed batches of training and validation text (reproducible estimate)."""
    model.eval()
    out = {}
    for name, d in (("train", train_data), ("val", val_data)):
        g = torch.Generator().manual_seed(1234)
        losses = [F.cross_entropy(model(x).flatten(0, 1), y.flatten()).item() for x, y in (get_batch(d, g) for _ in range(EVAL_BATCHES))]
        out[name] = float(np.mean(losses))
    model.train()
    return out


def train_phase(model: nn.Module, phase: str, train_data: torch.Tensor, val_data: torch.Tensor,
                steps: int, peak_lr: float, warmup: int = 200, eval_every: int = 250, seed: int = SEED) -> tuple[pd.DataFrame, dict]:
    """Train ``model`` with warm-up + cosine learning rate.

    Returns the loss history and the weights with the lowest validation loss seen at an
    evaluation step (early stopping); the model itself is left at its final weights.
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=peak_lr, weight_decay=0.1)
    g = torch.Generator().manual_seed(seed)
    history, start = [], time.time()
    best_val, best_state = float("inf"), None
    for step in range(steps + 1):
        lr = peak_lr * step / warmup if step < warmup else peak_lr * 0.5 * (1 + math.cos(math.pi * (step - warmup) / (steps - warmup)))
        for group in optimizer.param_groups:
            group["lr"] = lr
        if step % eval_every == 0:
            losses = estimate_loss(model, train_data, val_data)
            history.append({"phase": phase, "step": step, "train_loss": losses["train"], "val_loss": losses["val"], "lr": lr,
                            "minutes": (time.time() - start) / 60})
            print(f"[{phase}] step {step:5d}  train loss {losses['train']:.4f}  val loss {losses['val']:.4f}  lr {lr:.2e}")
            if losses["val"] < best_val:
                best_val, best_state = losses["val"], {k: v.detach().clone() for k, v in model.state_dict().items()}
        if step == steps:
            break
        x, y = get_batch(train_data, g)
        loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    return pd.DataFrame(history), best_state

### 4.1 Phase 1: pre-training on the whole Bible

In [ ]:
torch.manual_seed(SEED)
hist_pre, _ = train_phase(model, "pre-training", data["pretrain_train"], data["pretrain_val"], steps=4000, peak_lr=1e-3)
pretrained_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
print(f"pre-training time: {hist_pre.minutes.iloc[-1]:.1f} min")